# Mevsimsel Talep Belirsizliği Altında Akıllı Depo Yönetimi
## Klasik ve Derin Pekiştirmeli Öğrenme Yöntemlerinin Karşılaştırmalı Performans Analizi

**Nevin ÇAVUŞOĞLU — 25435021003**
Bursa Teknik Üniversitesi, Bilgisayar Mühendisliği Doktora Programı
Derin Pekiştirmeli Öğrenme Dersi — Final Projesi

Bu notebook üç deney koşulunu aynı mevsimsel depo ortamında eğitir ve karşılaştırır:
1. **Tablo Q (mevsimden habersiz)** — ablasyon koşulu, durum = (stok)
2. **Tablo Q (mevsim-farkında)** — klasik RL, durum = (stok, mevsim)
3. **DQN (mevsim-farkında)** — derin RL, durum = [stok/10, sin(2π·gün/365), cos(2π·gün/365)]

Mevsimsel talep parametreleri (λ), gerçek bir perakende veri setinden — **UCI Online Retail** (Chen vd., 2012; 531.285 işlem, Aralık 2010 – Aralık 2011) — kalibre edilmiştir. Kalibrasyon adımları 2. bölümde yeniden üretilebilir.

> Not: Tüm hücreler sırayla çalıştırılmalıdır (Runtime → Run all). Toplam süre CPU'da ~10 dk, GPU'da daha kısadır.

## 1. Kütüphaneler ve Rastgelelik Tohumları

In [ ]:
import numpy as np
import random
import torch
import matplotlib.pyplot as plt

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Kullanilan cihaz:", device)

## 2. Veri Seti ve Mevsimsel Talep Kalibrasyonu
Aşağıdaki hücre, UCI Online Retail veri setini indirir, temizler (iptal faturaları ve negatif miktarlar çıkarılır), günlük toplam satış adetlerini yılın çeyreklerine göre ortalar ve mevsimsel oranları koruyarak genel ortalama λ = 3.0 olacak şekilde tek-ürün ortam ölçeğine indirger. Bu hücre isteğe bağlıdır: internet erişimi yoksa atlanabilir; ortam sınıfı kalibre edilmiş değerleri zaten gömülü içerir.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv"
try:
    df = pd.read_csv(URL, dtype={"InvoiceNo": str})
    df = df[~df["InvoiceNo"].str.startswith("C", na=False)]
    df = df[df["Quantity"] > 0]
    df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], format="%m/%d/%Y %H:%M")
    daily = df.groupby([df["InvoiceDate"].dt.date, df["InvoiceDate"].dt.month])["Quantity"].sum()
    daily = daily.reset_index(level=1).rename(columns={"InvoiceDate": "month"})
    monthly_mean = daily.groupby("month")["Quantity"].mean()
    quarters = {"Kis (Oca-Mar)": [1,2,3], "Ilkbahar (Nis-Haz)": [4,5,6],
                "Yaz (Tem-Eyl)": [7,8,9], "Sonbahar (Eki-Ara)": [10,11,12]}
    qmeans = {k: monthly_mean[monthly_mean.index.isin(m)].mean() for k, m in quarters.items()}
    overall = sum(qmeans.values()) / 4
    print(f"Temiz islem sayisi: {len(df):,}")
    print(f"{'Ceyrek':22s} {'Gercek ort. talep':>18s} {'Oran':>7s} {'Kalibre lambda':>15s}")
    for k, v in qmeans.items():
        print(f"{k:22s} {v:15.0f} ad/g {v/overall:7.3f} {3.0*v/overall:15.2f}")
except Exception as e:
    print("Veri indirilemedi (internet gerekli), gomulu kalibre degerler kullanilacak:", e)

## 3. Mevsimsel Depo Ortamı
Günlük talep, mevsime bağlı Poisson dağılımından örneklenir; λ değerleri yukarıdaki kalibrasyondan gelir. Ödül: `r = satılan×10 − stok×2 − yoksatan×15 − sipariş×3`

In [ ]:
"""
Mevsimsel Akilli Depo Ortami
=============================
Sinirli kapasiteli bir depoda, gunluk talebin mevsime bagli bir Poisson
dagilimindan orneklendigi envanter yonetimi ortami.

  - Odul: r = satilan*10 - stok*2 - yoksatan*15 - siparis*3
  - Kapasite 10, max siparis 5, episode = 365 gun
  - Mevsim bilgisi state'e dahil edilir (Markov ozelligi korunur)
"""

import numpy as np

SEASON_NAMES = ["Kis", "Ilkbahar", "Yaz", "Sonbahar"]
DAYS_PER_YEAR = 365
# Gun -> mevsim esikleri (91+91+91+92 = 365)
SEASON_BOUNDS = [91, 182, 273, 365]

# Mevsimsel ortalama talep (Poisson lambda).
# UCI Online Retail veri setinden (Chen vd., 2012; Aralik 2010 - Aralik 2011,
# 531.285 islem) kalibre edilmistir: gunluk toplam satis adetleri yilin
# ceyreklerine gore ortalanmis, mevsimsel oranlar korunarak genel ortalama
# lambda = 3.0 olacak sekilde tek-urun ortam olcegine indirgenmistir.
# Gun araliklari: Kis=Oca-Mar, Ilkbahar=Nis-Haz, Yaz=Tem-Eyl, Sonbahar=Eki-Ara.
# Talep zirvesi, yilbasi alisveris donemi nedeniyle son ceyrektedir.
SEASON_LAMBDAS = {0: 2.32, 1: 2.50, 2: 2.94, 3: 4.24}


class MevsimselDepoOrtami:
    MAX_STOK = 10
    MAX_SIPARIS = 5
    SATIS_KARI = 10
    TUTMA_MALIYETI = 2
    YOKSATMA_CEZASI = 15
    SIPARIS_MALIYETI = 3

    def __init__(self, season_aware=True, seasonal_demand=True, seed=None):
        self.rng = np.random.default_rng(seed)
        self.season_aware = season_aware       # mevsim state'e dahil mi
        self.seasonal_demand = seasonal_demand # talep mevsimsel mi (False -> sabit lambda=3)
        self.stok = 0
        self.gun = 0
        self.n_actions = self.MAX_SIPARIS + 1
        self.state_dim = 3 if season_aware else 1  # [stok_norm, sin, cos] / [stok_norm]

    def _season_of_day(self, day):
        d = day % DAYS_PER_YEAR
        for i, b in enumerate(SEASON_BOUNDS):
            if d < b:
                return i
        return 3

    def reset(self):
        self.stok = int(self.rng.integers(0, self.MAX_STOK + 1))  # rastgele baslangic stogu
        self.gun = 0
        return self._get_state()

    def _get_state(self):
        stok_norm = self.stok / self.MAX_STOK
        if not self.season_aware:
            return np.array([stok_norm], dtype=np.float32)
        angle = 2 * np.pi * (self.gun % DAYS_PER_YEAR) / DAYS_PER_YEAR
        return np.array([stok_norm, np.sin(angle), np.cos(angle)], dtype=np.float32)

    def get_discrete_state(self):
        """Tabular Q-learning icin ayrik state."""
        if self.season_aware:
            return (self.stok, self._season_of_day(self.gun))
        return (self.stok,)

    def step(self, action):
        action = int(np.clip(action, 0, self.MAX_SIPARIS))
        # 1. siparis stoga eklenir (kapasite asilmaz)
        self.stok = min(self.stok + action, self.MAX_STOK)
        # 2. talep gerceklesir
        lam = SEASON_LAMBDAS[self._season_of_day(self.gun)] if self.seasonal_demand else 3.0
        talep = int(self.rng.poisson(lam))
        # 3. satis hesabi
        satilan = min(self.stok, talep)
        yoksatan = max(0, talep - self.stok)
        self.stok -= satilan
        # 4. odul hesabi
        reward = (satilan * self.SATIS_KARI
                  - self.stok * self.TUTMA_MALIYETI
                  - yoksatan * self.YOKSATMA_CEZASI
                  - action * self.SIPARIS_MALIYETI)
        self.gun += 1
        bitti = (self.gun >= DAYS_PER_YEAR)
        info = {"season": self._season_of_day(self.gun - 1), "talep": talep,
                "satilan": satilan, "yoksatan": yoksatan}
        return self._get_state(), reward, bitti, info

## 4. Klasik RL: Tablo Tabanlı Q-Öğrenme Ajanı

In [ ]:
"""
Tablo Tabanli Q-Learning Ajani (Klasik RL)
===========================================
Iki varyant desteklenir:
  1) Mevsimden habersiz (ablasyon)   -> state = (stok,)
  2) Mevsim-farkinda                 -> state = (stok, mevsim)
"""

import numpy as np


class TabularQAgent:
    def __init__(self, n_inventory, n_seasons, n_actions,
                 alpha=0.1, gamma=0.95, epsilon_start=1.0,
                 epsilon_end=0.05, epsilon_decay=0.9995, season_aware=True):
        self.n_actions = n_actions
        self.season_aware = season_aware
        shape = (n_inventory + 1, n_seasons, n_actions) if season_aware else (n_inventory + 1, n_actions)
        self.Q = np.zeros(shape)
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay

    def _idx(self, state):
        return state if self.season_aware else (state[0],)

    def select_action(self, state, greedy=False):
        if (not greedy) and (np.random.rand() < self.epsilon):
            return np.random.randint(self.n_actions)
        return int(np.argmax(self.Q[self._idx(state)]))

    def update(self, state, action, reward, next_state, done):
        idx = self._idx(state) + (action,)
        next_idx = self._idx(next_state)
        target = reward + (0.0 if done else self.gamma * np.max(self.Q[next_idx]))
        self.Q[idx] += self.alpha * (target - self.Q[idx])

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

## 5. Derin RL: Derin Q-Ağı (DQN) Ajanı
Deneyim tekrarı (replay buffer) + hedef ağ (target network) içerir.

In [ ]:
"""
Deep Q-Network (DQN) Agent
===========================
Final projesinin ana katkisi: Q-table yerine, state -> Q-degerleri
eslemesini ogrenen bir sinir agi (fonksiyon yaklastirici) kullanilir.

Bilesenler:
  - Deneyim tekrari havuzu (Experience Replay Buffer)
  - Hedef ag (Target Network) ile egitim kararliligi
  - Epsilon-greedy kesif (exploration) stratejisi
"""

import random
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim


class QNetwork(nn.Module):
    def __init__(self, state_dim, n_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, s2, done):
        self.buffer.append((s, a, r, s2, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, s2, done = zip(*batch)
        return (
            np.array(s, dtype=np.float32),
            np.array(a, dtype=np.int64),
            np.array(r, dtype=np.float32),
            np.array(s2, dtype=np.float32),
            np.array(done, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)


class DQNAgent:
    def __init__(
        self,
        state_dim,
        n_actions,
        lr=1e-3,
        gamma=0.95,
        epsilon_start=1.0,
        epsilon_end=0.05,
        epsilon_decay=0.9995,
        buffer_capacity=50000,
        batch_size=64,
        target_update_freq=200,
        device=None,
    ):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.n_actions = n_actions
        self.gamma = gamma
        self.epsilon = epsilon_start
        self.epsilon_end = epsilon_end
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        self.target_update_freq = target_update_freq
        self.step_count = 0

        self.policy_net = QNetwork(state_dim, n_actions).to(self.device)
        self.target_net = QNetwork(state_dim, n_actions).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_capacity)
        self.loss_fn = nn.MSELoss()

    def select_action(self, state, greedy=False):
        if (not greedy) and (random.random() < self.epsilon):
            return random.randint(0, self.n_actions - 1)
        with torch.no_grad():
            s = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            q = self.policy_net(s)
            return int(torch.argmax(q, dim=1).item())

    def store(self, s, a, r, s2, done):
        self.buffer.push(s, a, r, s2, done)

    def update(self):
        if len(self.buffer) < self.batch_size:
            return None
        s, a, r, s2, done = self.buffer.sample(self.batch_size)
        s = torch.tensor(s, device=self.device)
        a = torch.tensor(a, device=self.device).unsqueeze(1)
        r = torch.tensor(r, device=self.device)
        s2 = torch.tensor(s2, device=self.device)
        done = torch.tensor(done, device=self.device)

        q_values = self.policy_net(s).gather(1, a).squeeze(1)
        with torch.no_grad():
            next_q = self.target_net(s2).max(1)[0]
            target = r + self.gamma * next_q * (1 - done)

        loss = self.loss_fn(q_values, target)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.step_count += 1
        if self.step_count % self.target_update_freq == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())

        return loss.item()

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon_end, self.epsilon * self.epsilon_decay)

## 6. Eğitim ve Değerlendirme Fonksiyonları

In [ ]:
N_EP_TABULAR = 2000
N_EP_DQN = 300
CAP = 10; MAX_ORD = 5; N_SEASONS = 4


def run_tabular(season_aware, n_episodes=N_EP_TABULAR, seed=42):
    env = MevsimselDepoOrtami(season_aware=season_aware, seasonal_demand=True, seed=seed)
    agent = TabularQAgent(CAP, N_SEASONS, MAX_ORD + 1, alpha=0.1, gamma=0.99,
                          epsilon_start=1.0, epsilon_end=0.01, epsilon_decay=0.995,
                          season_aware=season_aware)
    ep_rewards = []
    for ep in range(n_episodes):
        env.reset()
        state = env.get_discrete_state()
        total, done = 0.0, False
        while not done:
            a = agent.select_action(state)
            _, r, done, _ = env.step(a)
            ns = env.get_discrete_state()
            agent.update(state, a, r, ns, done)
            state = ns; total += r
        agent.decay_epsilon()
        ep_rewards.append(total)
    return np.array(ep_rewards), agent


def run_dqn(n_episodes=N_EP_DQN, seed=42):
    env = MevsimselDepoOrtami(season_aware=True, seasonal_demand=True, seed=seed)
    agent = DQNAgent(state_dim=env.state_dim, n_actions=env.n_actions, gamma=0.99,
                     epsilon_decay=0.985, epsilon_end=0.01)
    ep_rewards = []
    for ep in range(n_episodes):
        state = env.reset()
        total, done = 0.0, False
        while not done:
            a = agent.select_action(state)
            ns, r, done, _ = env.step(a)
            agent.store(state, a, r, ns, float(done))
            agent.update()
            state = ns; total += r
        agent.decay_epsilon()
        ep_rewards.append(total)
        if (ep + 1) % 50 == 0:
            print(f"  DQN episode {ep+1}/{n_episodes}  son-20 ort: {np.mean(ep_rewards[-20:]):.0f}  eps: {agent.epsilon:.3f}")
    return np.array(ep_rewards), agent


def moving_avg(x, w):
    return np.convolve(x, np.ones(w) / w, mode="valid")


def evaluate(agent, kind, n_episodes=30, seed=999):
    """Egitim sonrasi greedy (eps=0) politika ile bagimsiz test yillari."""
    env = MevsimselDepoOrtami(season_aware=(kind != "tab_no_season"),
                              seasonal_demand=True, seed=seed)
    rewards = []
    for _ in range(n_episodes):
        sv = env.reset()
        s = sv if kind == "dqn" else env.get_discrete_state()
        total, done = 0.0, False
        while not done:
            a = agent.select_action(s, greedy=True)
            sv, r, done, _ = env.step(a)
            s = sv if kind == "dqn" else env.get_discrete_state()
            total += r
        rewards.append(total)
    return np.array(rewards)

## 7. Eğitim
Üç koşul sırayla eğitilir. Tablo ajanları hızlıdır; DQN, adım başına mini-batch güncellemesi yaptığı için birkaç dakika sürer.

In [ ]:
print("[1/3] Tablo Q (mevsimden habersiz - ablasyon)...")
r_tab1, ag_tab1 = run_tabular(season_aware=False)
print("      tamamlandi. Son-50 episode ort:", r_tab1[-50:].mean().round(1))

print("[2/3] Tablo Q (mevsim-farkinda)...")
r_tab2, ag_tab2 = run_tabular(season_aware=True)
print("      tamamlandi. Son-50 episode ort:", r_tab2[-50:].mean().round(1))

print("[3/3] DQN (mevsim-farkinda)...")
r_dqn, ag_dqn = run_dqn()
print("      tamamlandi. Son-30 episode ort:", r_dqn[-30:].mean().round(1))

## 8. Öğrenme Eğrileri (Şekil 1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(moving_avg(r_tab1, 50), label="Tablo Q (mevsimden habersiz)", lw=1.6)
axes[0].plot(moving_avg(r_tab2, 50), label="Tablo Q (mevsim-farkinda)", lw=1.6)
axes[0].set_title("(a) Tablo tabanli ajanlar (2000 episode)")
axes[0].set_xlabel("Episode (50-ep hareketli ort.)"); axes[0].set_ylabel("Toplam Yillik Odul")
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(moving_avg(r_dqn, 20), color="#2e8b57", lw=1.8, label="DQN (mevsim-farkinda)")
axes[1].set_title("(b) DQN (300 episode)")
axes[1].set_xlabel("Episode (20-ep hareketli ort.)"); axes[1].set_ylabel("Toplam Yillik Odul")
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("learning_curves.png", dpi=150)
plt.show()

## 9. Test Değerlendirmesi ve Karşılaştırma (Tablo III, Şekil 2)
30 bağımsız test yılı, greedy politika (ε = 0).

In [ ]:
e1 = evaluate(ag_tab1, "tab_no_season")
e2 = evaluate(ag_tab2, "tab_season")
e3 = evaluate(ag_dqn, "dqn")

results = {
    "Tablo Q (mevsimden habersiz)": e1,
    "Tablo Q (mevsim-farkinda)": e2,
    "DQN (mevsim-farkinda)": e3,
}
print(f"{'Kosul':32s} {'Ort.':>8s} {'Std':>7s} {'Min':>7s} {'Max':>7s}")
for name, arr in results.items():
    print(f"{name:32s} {arr.mean():8.1f} {arr.std():7.1f} {arr.min():7.0f} {arr.max():7.0f}")

base = e1.mean()
print(f"\nMevsim bilgisinin katkisi (temsil etkisi): %{100*(e2.mean()-base)/base:.1f}")
print(f"DQN'in tablo yontemine gore ek katkisi (yontem etkisi): %{100*(e3.mean()-e2.mean())/e2.mean():.1f}")
print(f"Toplam iyilesme: %{100*(e3.mean()-base)/base:.1f}")

plt.figure(figsize=(7, 5))
names = list(results.keys())
means = [results[k].mean() for k in names]
stds = [results[k].std() for k in names]
plt.bar(names, means, yerr=stds, capsize=6, color=["#8899aa", "#d9a441", "#3f8f5f"])
plt.ylabel("Ortalama Yillik Odul")
plt.title("Test Performansi (30 yil, greedy politika)")
plt.xticks(rotation=15, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("comparison_bar.png", dpi=150)
plt.show()

## 10. DQN'in Öğrendiği Mevsimsel Politika (Şekil 3)

In [ ]:
grid = np.zeros((4, CAP + 1))
for si, day in enumerate([45, 136, 227, 319]):  # her mevsimin ortasi
    for inv in range(CAP + 1):
        ang = 2 * np.pi * day / DAYS_PER_YEAR
        s = np.array([inv / CAP, np.sin(ang), np.cos(ang)], dtype=np.float32)
        grid[si, inv] = ag_dqn.select_action(s, greedy=True)

plt.figure(figsize=(8, 4.5))
im = plt.imshow(grid, aspect="auto", cmap="viridis", vmin=0, vmax=MAX_ORD)
plt.colorbar(im, label="Ogrenilen Siparis Miktari")
plt.yticks(range(4), SEASON_NAMES)
plt.xticks(range(CAP + 1))
plt.xlabel("Mevcut Stok Seviyesi")
plt.ylabel("Mevsim")
plt.title("DQN Ajaninin Ogrendigi Mevsimsel Siparis Politikasi")
plt.tight_layout()
plt.savefig("policy_heatmap.png", dpi=150)
plt.show()

print("Politika matrisi (satir: mevsim, sutun: stok):")
print(grid.astype(int))

## 11. Simülasyon GIF'i
Eğitilmiş DQN ajanının 365 günlük bir yılı greedy politikayla oynadığı animasyon. Hücre sonunda GIF notebook içinde görüntülenir ve `dqn_simulasyon.gif` olarak kaydedilir (Colab'da sol paneldeki Dosyalar sekmesinden indirilebilir).

In [ ]:
from matplotlib.patches import Rectangle, FancyArrow
from PIL import Image
import io

SEASON_NAMES_TR = ["KIS", "ILKBAHAR", "YAZ", "SONBAHAR"]
SEASON_COLORS = ["#4a90d9", "#5fb96a", "#e8b73a", "#c96f42"]
BG = "#0f1117"; PANEL = "#1a1a2e"; GRID = "#2a2a3e"
TXT = "#e8e8e8"; ACCENT = "#00d4ff"; REWARD_C = "#ff6b6b"

# --- 1 yillik greedy simulasyon kaydi ---
sim_env = MevsimselDepoOrtami(season_aware=True, seasonal_demand=True, seed=777)
state = sim_env.reset()
log, cum, done = [], 0.0, False
while not done:
    gun, stok_once = sim_env.gun, sim_env.stok
    a = ag_dqn.select_action(state, greedy=True)
    state, r, done, info = sim_env.step(a)
    cum += r
    log.append(dict(gun=gun + 1, mevsim=info["season"], siparis=a, talep=info["talep"],
                    satilan=info["satilan"], yoksatan=info["yoksatan"],
                    stok_sonra=sim_env.stok, odul=r, kumulatif=cum))
print(f"Simulasyon yillik toplam odulu: {cum:.0f}")

gunler = [d["gun"] for d in log]
stoklar = [d["stok_sonra"] for d in log]
kumulatifler = [d["kumulatif"] for d in log]


def draw_frame(i):
    d = log[i]
    fig = plt.figure(figsize=(9.6, 5.4), dpi=72)
    fig.patch.set_facecolor(BG)
    gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 1.9], height_ratios=[1, 1],
                          hspace=0.45, wspace=0.28,
                          left=0.06, right=0.97, top=0.86, bottom=0.10)
    sc = SEASON_COLORS[d["mevsim"]]
    fig.suptitle(f"Akilli Depo - DQN Simulasyonu | Gun {d['gun']:3d}/365 | "
                 f"{SEASON_NAMES_TR[d['mevsim']]} (lam={SEASON_LAMBDAS[d['mevsim']]})",
                 color=TXT, fontsize=13, fontweight="bold", y=0.965)
    fig.patches.append(Rectangle((0.0, 0.90), 1.0, 0.012, transform=fig.transFigure,
                                  facecolor=sc, edgecolor="none"))
    ax1 = fig.add_subplot(gs[:, 0]); ax1.set_facecolor(PANEL)
    ax1.set_xlim(0, 10); ax1.set_ylim(-0.8, 11.5); ax1.set_xticks([]); ax1.set_yticks([])
    for sp in ax1.spines.values(): sp.set_edgecolor(GRID)
    ax1.set_title("Depo Durumu", color=TXT, fontsize=10, fontweight="bold")
    ax1.add_patch(Rectangle((2.6, 0), 4.8, 10.6, fill=False, edgecolor="#888", lw=2))
    for k in range(d["stok_sonra"]):
        ax1.add_patch(Rectangle((2.85, k * 1.05 + 0.1), 4.3, 0.85,
                                facecolor=ACCENT, edgecolor=BG, lw=1))
    ax1.text(5.0, 11.0, f"Stok: {d['stok_sonra']}/10", color=ACCENT, fontsize=11,
             ha="center", fontweight="bold")
    if d["siparis"] > 0:
        ax1.add_patch(FancyArrow(0.4, 8.6, 1.6, 0, width=0.32, head_width=0.75,
                                 head_length=0.45, facecolor="#5fb96a", edgecolor="none"))
        ax1.text(1.2, 9.5, f"Siparis +{d['siparis']}", color="#5fb96a", fontsize=9, ha="center")
    ax1.add_patch(FancyArrow(7.7, 2.0, 1.6, 0, width=0.32, head_width=0.75,
                             head_length=0.45, facecolor="#e8b73a", edgecolor="none"))
    ax1.text(8.6, 2.95, f"Talep {d['talep']}", color="#e8b73a", fontsize=9, ha="center")
    if d["yoksatan"] > 0:
        ax1.text(8.6, 1.0, f"Yoksatma: {d['yoksatan']}", color=REWARD_C, fontsize=9,
                 ha="center", fontweight="bold")
    ax1.text(5.0, -0.55, f"Satilan: {d['satilan']}   Gunluk odul: {d['odul']:+.0f}",
             color=TXT, fontsize=9, ha="center")
    ax2 = fig.add_subplot(gs[0, 1]); ax2.set_facecolor(PANEL)
    for sp in ax2.spines.values(): sp.set_edgecolor(GRID)
    ax2.grid(color=GRID, lw=0.5, alpha=0.7); ax2.tick_params(colors="#aaaaaa", labelsize=7)
    prev = 0
    for si, b in enumerate(SEASON_BOUNDS):
        ax2.axvspan(prev + 1, b, color=SEASON_COLORS[si], alpha=0.10); prev = b
    ax2.plot(gunler[:i + 1], stoklar[:i + 1], color=ACCENT, lw=1.4)
    ax2.plot(d["gun"], d["stok_sonra"], "o", color="white", ms=5)
    ax2.set_xlim(1, 365); ax2.set_ylim(-0.5, 10.5)
    ax2.set_title("Stok Seviyesi (gun sonu)", color=TXT, fontsize=10, fontweight="bold")
    ax2.set_ylabel("Stok", color="#aaaaaa", fontsize=8)
    ax3 = fig.add_subplot(gs[1, 1]); ax3.set_facecolor(PANEL)
    for sp in ax3.spines.values(): sp.set_edgecolor(GRID)
    ax3.grid(color=GRID, lw=0.5, alpha=0.7); ax3.tick_params(colors="#aaaaaa", labelsize=7)
    prev = 0
    for si, b in enumerate(SEASON_BOUNDS):
        ax3.axvspan(prev + 1, b, color=SEASON_COLORS[si], alpha=0.10); prev = b
    ax3.plot(gunler[:i + 1], kumulatifler[:i + 1], color=REWARD_C, lw=1.4)
    ax3.set_xlim(1, 365)
    ax3.set_ylim(min(0, min(kumulatifler)) - 100, max(kumulatifler) * 1.05)
    ax3.set_title(f"Kumulatif Odul: {d['kumulatif']:.0f}", color=TXT, fontsize=10, fontweight="bold")
    ax3.set_xlabel("Gun", color="#aaaaaa", fontsize=8)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", facecolor=BG)
    plt.close(fig)
    buf.seek(0)
    return Image.open(buf).convert("P", palette=Image.ADAPTIVE, colors=128)


frame_days = list(range(0, DAYS_PER_YEAR, 2))
if (DAYS_PER_YEAR - 1) not in frame_days:
    frame_days.append(DAYS_PER_YEAR - 1)
frames = []
for n, i in enumerate(frame_days):
    frames.append(draw_frame(i))
    if (n + 1) % 40 == 0:
        print(f"  kare {n+1}/{len(frame_days)}")
durations = [70] * len(frames); durations[-1] = 2500
frames[0].save("dqn_simulasyon.gif", save_all=True, append_images=frames[1:],
               duration=durations, loop=0, optimize=True)
import os
print(f"GIF kaydedildi: dqn_simulasyon.gif ({os.path.getsize('dqn_simulasyon.gif')/1e6:.1f} MB)")

from IPython.display import Image as IPImage, display
display(IPImage(filename="dqn_simulasyon.gif"))